In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# Only the final trained weights go to Drive — nothing else.
FINAL_WEIGHTS_DIR = '/content/drive/MyDrive/TransGeo_CVIAN_final'
os.makedirs(FINAL_WEIGHTS_DIR, exist_ok=True)

print('Final weights will be saved to:', FINAL_WEIGHTS_DIR)

In [ ]:
!git clone https://github.com/Jeff-Zilence/TransGeo2022.git /content/TransGeo2022
%cd /content/TransGeo2022

!pip install -q timm ptflops einops

In [ ]:
%%writefile dataset/CVIAN.py
import torch
import torchvision.transforms as transforms
from PIL import Image
import os

# CVIAN dataset loader for TransGeo.
# Mirrors dataset/CVUSA.py's interface so it plugs straight into train.py.
#
# Expects the CVIAN folder produced by CVDisaster's 0_prepare_splits_and_data.py
# (https://github.com/tum-bgd/CVDisaster), i.e. a root directory containing:
#   00_SVI_aligned/...        (street-view / ground panoramas, compass-aligned)
#   01_Satellite/...          (satellite tiles)
#   train_<split>.csv         (two columns, no header: sat_path,ground_path)
#   val_<split>.csv
#
# <split> defaults to "subarea5" to match the split Ruixi already used for
# Sample4Geo/CVDisaster, so the TransGeo vs Sample4Geo comparison is
# trained/evaluated on identical data.


def input_transform(size):
    return transforms.Compose([
        transforms.Resize(size=tuple(size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                              std=[0.229, 0.224, 0.225]),
    ])


class CVIAN(torch.utils.data.Dataset):
    def __init__(self, mode='', root='', same_area=True, print_bool=False, polar='', args=None):
        super(CVIAN, self).__init__()

        self.args = args
        self.mode = mode

        if root:
            self.root = root
        elif args is not None and getattr(args, 'root', ''):
            self.root = args.root
        else:
            self.root = '/content/CVDisaster/geolocalization/CVIAN/'
        if not self.root.endswith('/'):
            self.root += '/'

        self.sat_size = [256, 256]
        if args is not None and getattr(args, 'sat_res', 0) != 0:
            self.sat_size = [args.sat_res, args.sat_res]
        self.grd_size = [112, 616]

        self.transform_query = input_transform(size=self.grd_size)
        self.transform_reference = input_transform(size=self.sat_size)
        self.to_tensor = transforms.ToTensor()

        split_key = getattr(args, 'cvian_split', 'subarea5') if args is not None else 'subarea5'
        self.train_list = self.root + f'train_{split_key}.csv'
        self.test_list = self.root + f'val_{split_key}.csv'

        if print_bool:
            print('CVIAN: load %s' % self.train_list)

        self.id_list = []
        self.id_idx_list = []
        with open(self.train_list, 'r') as f:
            for idx, line in enumerate(f):
                line = line.strip()
                if not line:
                    continue
                data = line.split(',')
                pano_id = os.path.splitext(os.path.basename(data[0]))[0]
                self.id_list.append([data[0], data[1], pano_id])
                self.id_idx_list.append(idx)
        self.data_size = len(self.id_list)
        if print_bool:
            print('CVIAN: load', self.train_list, ' data_size =', self.data_size)
            print('CVIAN: load %s' % self.test_list)

        self.id_test_list = []
        self.id_test_idx_list = []
        with open(self.test_list, 'r') as f:
            for idx, line in enumerate(f):
                line = line.strip()
                if not line:
                    continue
                data = line.split(',')
                pano_id = os.path.splitext(os.path.basename(data[0]))[0]
                self.id_test_list.append([data[0], data[1], pano_id])
                self.id_test_idx_list.append(idx)
        self.test_data_size = len(self.id_test_list)
        if print_bool:
            print('CVIAN: load', self.test_list, ' data_size =', self.test_data_size)

    def __getitem__(self, index):
        if self.mode == 'train':
            idx = index % len(self.id_idx_list)
            img_query = Image.open(self.root + self.id_list[idx][1]).convert('RGB')
            img_reference = Image.open(self.root + self.id_list[idx][0]).convert('RGB')
            img_query = self.transform_query(img_query)
            img_reference = self.transform_reference(img_reference)
            return img_query, img_reference, torch.tensor(idx), torch.tensor(idx), 0, 0

        elif 'scan_val' in self.mode:
            img_reference = Image.open(self.root + self.id_test_list[index][0]).convert('RGB')
            img_reference = self.transform_reference(img_reference)
            img_query = Image.open(self.root + self.id_test_list[index][1]).convert('RGB')
            img_query = self.transform_query(img_query)
            return img_query, img_reference, torch.tensor(index), torch.tensor(index), 0, 0

        elif 'test_reference' in self.mode:
            img_reference = Image.open(self.root + self.id_test_list[index][0]).convert('RGB')
            img_reference = self.transform_reference(img_reference)
            return img_reference, torch.tensor(index), 0

        elif 'test_query' in self.mode:
            img_query = Image.open(self.root + self.id_test_list[index][1]).convert('RGB')
            img_query = self.transform_query(img_query)
            return img_query, torch.tensor(index), torch.tensor(index)

        else:
            print('not implemented!!')
            raise Exception

    def __len__(self):
        if self.mode == 'train':
            return len(self.id_idx_list)
        elif 'scan_val' in self.mode:
            return len(self.id_test_list)
        elif 'test_reference' in self.mode:
            return len(self.id_test_list)
        elif 'test_query' in self.mode:
            return len(self.id_test_list)
        else:
            print('not implemented!')
            raise Exception

In [ ]:
with open('train.py', 'r') as f:
    code = f.read()

code = code.replace(
    "from dataset.CVACT import CVACT\n",
    "from dataset.CVACT import CVACT\nfrom dataset.CVIAN import CVIAN\n"
)

code = code.replace(
    "parser.add_argument('--dataset', default='vigor', type=str,\n"
    "                    help='vigor, cvusa, cvact')\n",
    "parser.add_argument('--dataset', default='vigor', type=str,\n"
    "                    help='vigor, cvusa, cvact, cvian')\n"
    "parser.add_argument('--root', default='', type=str,\n"
    "                    help='dataset root path (used by CVIAN loader)')\n"
    "parser.add_argument('--cvian_split', default='subarea5', type=str,\n"
    "                    help='CVIAN split key, e.g. subarea5 / val60 (matches CVDisaster split_key)')\n"
)

code = code.replace(
    "    elif args.dataset == 'cvact':\n"
    "        size_sat = [256, 256]  # [512, 512]\n"
    "        size_sat_default = [256, 256]  # [512, 512]\n"
    "        size_grd = [112, 616]  # [224, 1232]\n"
    "\n"
    "    if args.sat_res != 0:",
    "    elif args.dataset == 'cvact':\n"
    "        size_sat = [256, 256]  # [512, 512]\n"
    "        size_sat_default = [256, 256]  # [512, 512]\n"
    "        size_grd = [112, 616]  # [224, 1232]\n"
    "    elif args.dataset == 'cvian':\n"
    "        size_sat = [256, 256]\n"
    "        size_sat_default = [256, 256]\n"
    "        size_grd = [112, 616]\n"
    "\n"
    "    if args.sat_res != 0:"
)

code = code.replace(
    "    elif args.dataset == 'cvact':\n"
    "        dataset = CVACT\n"
    "        mining_sampler = DistributedMiningSampler\n",
    "    elif args.dataset == 'cvact':\n"
    "        dataset = CVACT\n"
    "        mining_sampler = DistributedMiningSampler\n"
    "    elif args.dataset == 'cvian':\n"
    "        dataset = CVIAN\n"
    "        mining_sampler = DistributedMiningSampler\n"
)

with open('train.py', 'w') as f:
    f.write(code)

import ast
ast.parse(code)
print('train.py patched OK')

In [ ]:
%%writefile run_CVIAN.sh
#!/usr/bin/env bash
ROOT=${1:-/content/CVDisaster/geolocalization/CVIAN/}
SAVE_PATH=${2:-/content/result_cvian}
MODE=${3:-}
CKPT=${4:-}

CKPT_ARG=""
if [ "$MODE" = "--finetune" ] && [ -n "$CKPT" ]; then
  CKPT_ARG="--finetune $CKPT"
elif [ "$MODE" = "--resume" ] && [ -n "$CKPT" ]; then
  CKPT_ARG="--resume $CKPT"
fi

python -u train.py --workers 4 --lr 0.0001 --batch-size 16 --dist-url 'tcp://localhost:10001' \
  --multiprocessing-distributed --world-size 1 --rank 0 --epochs 10 \
  --save_path "$SAVE_PATH" --op sam --wd 0.03 --mining --dataset cvian \
  --root "$ROOT" --cvian_split subarea5 --cos --dim 1000 --asam --rho 2.5 \
  $CKPT_ARG

In [ ]:
!git clone https://github.com/tum-bgd/CVDisaster.git /content/CVDisaster
%cd /content/CVDisaster/geolocalization

!wget "https://dataserv.ub.tum.de/index.php/s/m1749324/download" -O m1749324.zip
!unzip -q m1749324.zip -d .

# The downloaded archive already extracts into a folder literally named CVIAN.
!mv m1749324/CVIAN ./CVIAN
!rm -f m1749324.zip
!rm -rf m1749324

!python 0_prepare_splits_and_data.py

In [ ]:
%cd /content/TransGeo2022

!pip install -q gdown
!gdown --folder https://drive.google.com/drive/folders/1oaITF_WJojkNaCMGTkBwNjOASEIEg6Vr \
  -O /content/pretrained

!tar -xzf /content/pretrained/CVUSA_model.tar.gz -C /content/pretrained

!find /content/pretrained -iname "*.pth*"

In [ ]:
with open('model/TransGeo.py', 'r') as f:
    code = f.read()

old = """        elif args.dataset == 'cvact':
            self.size_sat = [256, 256]
            self.size_sat_default = [256, 256]
            self.size_grd = [112, 616]

        if args.sat_res != 0:"""

new = """        elif args.dataset == 'cvact':
            self.size_sat = [256, 256]
            self.size_sat_default = [256, 256]
            self.size_grd = [112, 616]
        elif args.dataset == 'cvian':
            self.size_sat = [256, 256]
            self.size_sat_default = [256, 256]
            self.size_grd = [112, 616]

        if args.sat_res != 0:"""

assert old in code, 'anchor not found — check model/TransGeo.py was not already patched'
code = code.replace(old, new)

with open('model/TransGeo.py', 'w') as f:
    f.write(code)

import ast
ast.parse(code)
print('model/TransGeo.py patched OK')

In [ ]:
with open('train.py', 'r') as f:
    code = f.read()

old1 = 'checkpoint = torch.load(args.resume)'
new1 = 'checkpoint = torch.load(args.resume, weights_only=False)'
assert code.count(old1) == 1
code = code.replace(old1, new1)

old2 = "checkpoint = torch.load(args.resume, map_location=loc)"
new2 = "checkpoint = torch.load(args.resume, map_location=loc, weights_only=False)"
assert code.count(old2) == 1
code = code.replace(old2, new2)

with open('train.py', 'w') as f:
    f.write(code)

import ast
ast.parse(code)
print('train.py patched OK (weights_only fix)')

In [ ]:
with open('train.py', 'r') as f:
    code = f.read()

old1 = """            if not args.crop:
                args.start_epoch = checkpoint['epoch']
            best_acc1 = checkpoint['best_acc1']"""
new1 = """            if not args.crop and args.dataset != 'cvian':
                args.start_epoch = checkpoint['epoch']
            if args.dataset != 'cvian':
                best_acc1 = checkpoint['best_acc1']"""
assert code.count(old1) == 1
code = code.replace(old1, new1)

old2 = "            model.load_state_dict(checkpoint['state_dict'])\n"
new2 = """            msg = model.load_state_dict(checkpoint['state_dict'], strict=False)
            print('Loaded pretrained weights with strict=False (mismatched keys kept at random init):')
            print(msg)
"""
assert code.count(old2) == 1
code = code.replace(old2, new2)

old3 = "            if args.op == 'sam' and args.dataset != 'cvact':    # Loading the optimizer status gives better result on CVUSA, but not on CVACT.\n                optimizer.load_state_dict(checkpoint['optimizer'])"
new3 = "            if args.op == 'sam' and args.dataset not in ('cvact', 'cvian'):    # Loading the optimizer status gives better result on CVUSA, but not on CVACT/CVIAN.\n                optimizer.load_state_dict(checkpoint['optimizer'])"
assert code.count(old3) == 1
code = code.replace(old3, new3)

with open('train.py', 'w') as f:
    f.write(code)

import ast
ast.parse(code)
print('train.py patched OK (resume-as-init fix for cvian)')

In [ ]:
with open('train.py', 'r') as f:
    code = f.read()

old = """            msg = model.load_state_dict(checkpoint['state_dict'], strict=False)
            print('Loaded pretrained weights with strict=False (mismatched keys kept at random init):')
            print(msg)
"""
new = """            pretrained_state = checkpoint['state_dict']
            current_state = model.state_dict()
            compatible_state = {
                k: v for k, v in pretrained_state.items()
                if k in current_state and v.shape == current_state[k].shape
            }
            skipped = [k for k in pretrained_state if k not in compatible_state]
            msg = model.load_state_dict(compatible_state, strict=False)
            print('Loaded pretrained weights; skipped due to shape mismatch:', skipped)
            print(msg)
"""
assert code.count(old) == 1
code = code.replace(old, new)

with open('train.py', 'w') as f:
    f.write(code)

import ast
ast.parse(code)
print('train.py patched OK (shape-mismatch-safe checkpoint loading)')

In [ ]:
with open('dataset/global_sampler.py', 'r') as f:
    code = f.read()

old = "np.zeros(self.queue_length,dtype=np.int)"
new = "np.zeros(self.queue_length,dtype=int)"
count = code.count(old)
assert count == 2, count
code = code.replace(old, new)

with open('dataset/global_sampler.py', 'w') as f:
    f.write(code)

import ast
ast.parse(code)
print(f'dataset/global_sampler.py patched OK ({count} occurrences fixed)')

In [ ]:
with open('dataset/global_sampler.py', 'r') as f:
    code = f.read()

old = "ranking[i] = product_train.shape[1] - 1 - np.where(mining_pool[product_index[i]] == i)[0]"
new = "ranking[i] = product_train.shape[1] - 1 - np.where(mining_pool[product_index[i]] == i)[0][0]"
assert code.count(old) == 1
code = code.replace(old, new)

with open('dataset/global_sampler.py', 'w') as f:
    f.write(code)

import ast
ast.parse(code)
print('dataset/global_sampler.py patched OK (scalar-extraction fix for numpy)')

In [ ]:
with open('dataset/global_sampler.py', 'r') as f:
    code = f.read()

old = "                ranking[i] = product_train.shape[1] - 1 - np.where(mining_pool[product_index[i]] == i)[0][0]"
new = """                match = np.where(mining_pool[product_index[i]] == i)[0]
                if len(match) > 0:
                    ranking[i] = product_train.shape[1] - 1 - match[0]"""
assert code.count(old) == 1
code = code.replace(old, new)

with open('dataset/global_sampler.py', 'w') as f:
    f.write(code)

import ast
ast.parse(code)
print('dataset/global_sampler.py patched OK (handles no-match case safely)')

In [ ]:
with open('train.py', 'r') as f:
    code = f.read()

old = """            train_sampler = mining_sampler(train_dataset, batch_size=args.batch_size, dim=args.dim, save_path=args.save_path)
            if args.resume:
                train_sampler.load(args.resume.replace(args.resume.split('/')[-1],''))"""
new = """            train_sampler = mining_sampler(train_dataset, batch_size=args.batch_size, dim=args.dim, save_path=args.save_path)
            if args.resume and args.dataset != 'cvian':
                train_sampler.load(args.resume.replace(args.resume.split('/')[-1],''))"""
assert code.count(old) == 1
code = code.replace(old, new)

with open('train.py', 'w') as f:
    f.write(code)

import ast
ast.parse(code)
print('train.py patched OK (skip loading foreign CVUSA mining queue for cvian)')

In [ ]:
import re

with open('train.py', 'r') as f:
    content = f.read()

# 1) --resume / --finetune argparse args
if '--finetune' not in content:
    old_resume_arg = """parser.add_argument('--resume', default='', type=str, metavar='PATH',
                    help='path to latest checkpoint (default: none)')"""
    new_resume_arg = """parser.add_argument('--resume', default='', type=str, metavar='PATH',
                    help='path to latest checkpoint of THIS SAME run/dataset '
                         '(true resume: inherits epoch, best_acc1, optimizer '
                         'state, and mining-queue state)')
parser.add_argument('--finetune', default='', type=str, metavar='PATH',
                    help='path to a checkpoint from a DIFFERENT dataset '
                         '(e.g. CVUSA) used only to initialize model weights '
                         '(shape-filtered). Does not affect epoch/best_acc1/'
                         'optimizer/mining-queue state.')"""
    assert old_resume_arg in content, "resume arg block not found"
    content = content.replace(old_resume_arg, new_resume_arg)
else:
    print("--finetune arg already present, skipping step 1")

# 2) checkpoint-loading block start
if 'ckpt_path = args.resume if args.resume else args.finetune' not in content:
    old_block_start = """    # optionally resume from a checkpoint
    if args.resume:
        if os.path.isfile(args.resume):
            print("=> loading checkpoint '{}'".format(args.resume))
            if args.gpu is None:
                checkpoint = torch.load(args.resume, weights_only=False)
            else:
                # Map model to be loaded to specified single gpu.
                loc = 'cuda:{}'.format(args.gpu)
                checkpoint = torch.load(args.resume, map_location=loc, weights_only=False)
            if not args.crop and args.dataset != 'cvian':
                args.start_epoch = checkpoint['epoch']
            if args.dataset != 'cvian':
                best_acc1 = checkpoint['best_acc1']
            if args.crop and args.sat_res != 0:"""
    new_block_start = """    # optionally resume/finetune from a checkpoint.
    # --resume   = true resume of THIS run on THIS dataset: inherit epoch,
    #              best_acc1, optimizer state, and (later below) mining-queue
    #              state, since the checkpoint's shapes/sizes match exactly.
    # --finetune = initialize weights from a DIFFERENT dataset's checkpoint
    #              (e.g. the CVUSA pretrained model): only the (shape-filtered)
    #              model weights are used; epoch/best_acc1/optimizer/mining
    #              queue are all left at their fresh-start defaults, since
    #              none of that state is valid across datasets.
    ckpt_path = args.resume if args.resume else args.finetune
    is_true_resume = bool(args.resume)
    if ckpt_path:
        if os.path.isfile(ckpt_path):
            print("=> loading checkpoint '{}'".format(ckpt_path))
            if args.gpu is None:
                checkpoint = torch.load(ckpt_path, weights_only=False)
            else:
                # Map model to be loaded to specified single gpu.
                loc = 'cuda:{}'.format(args.gpu)
                checkpoint = torch.load(ckpt_path, map_location=loc, weights_only=False)
            if not args.crop and is_true_resume:
                args.start_epoch = checkpoint['epoch']
            if is_true_resume:
                best_acc1 = checkpoint['best_acc1']
            if args.crop and args.sat_res != 0:"""
    assert old_block_start in content, "checkpoint-loading block start not found"
    content = content.replace(old_block_start, new_block_start)
else:
    print("checkpoint-loading block already patched, skipping step 2")

# 3) tail (optimizer load + final print statements)
if "print('Loaded pretrained weights; skipped due to shape mismatch:', skipped)\n            print(msg)\n            # Only load optimizer state" not in content:
    old_tail_variants = [
        """            skipped = [k for k in pretrained_state if k not in compatible_state]
            msg = model.load_state_dict(compatible_state, strict=False)
            print('Loaded pretrained weights; skipped due to shape mismatch:', skipped)
            print(msg)
            if args.op == 'sam' and args.dataset not in ('cvact', 'cvian'):    # Loading the optimizer status gives better result on CVUSA, but not on CVACT/CVIAN.
                optimizer.load_state_dict(checkpoint['optimizer'])
            print("=> loaded checkpoint '{}' (epoch {})"
                  .format(args.resume, checkpoint['epoch']))
        else:
            print("=> no checkpoint found at '{}'".format(args.resume))""",
    ]
    new_tail = """            skipped = [k for k in pretrained_state if k not in compatible_state]
            msg = model.load_state_dict(compatible_state, strict=False)
            print('Loaded pretrained weights; skipped due to shape mismatch:', skipped)
            print(msg)
            # Only load optimizer state on a true same-dataset resume; a
            # cross-dataset --finetune checkpoint's optimizer state (step
            # counts, moment estimates) is meaningless here.
            if args.op == 'sam' and is_true_resume and args.dataset not in ('cvact',):
                optimizer.load_state_dict(checkpoint['optimizer'])
            print("=> loaded checkpoint '{}' (epoch {})"
                  .format(ckpt_path, checkpoint['epoch']))
        else:
            print("=> no checkpoint found at '{}'".format(ckpt_path))"""
    found = False
    for variant in old_tail_variants:
        if variant in content:
            content = content.replace(variant, new_tail)
            found = True
            break
    assert found, "tail block not found (check train.py manually around model.load_state_dict)"
else:
    print("tail block already patched, skipping step 3")

# 4) mining-queue load — handle either the original or the earlier
#    root-cause-fixed ("and args.dataset != 'cvian'") version.
if 'Only reload the mining queue on a true same-dataset resume' not in content:
    old_sampler_variants = [
        """            train_sampler = mining_sampler(train_dataset, batch_size=args.batch_size, dim=args.dim, save_path=args.save_path)
            if args.resume:
                train_sampler.load(args.resume.replace(args.resume.split('/')[-1],''))""",
        """            train_sampler = mining_sampler(train_dataset, batch_size=args.batch_size, dim=args.dim, save_path=args.save_path)
            if args.resume and args.dataset != 'cvian':
                train_sampler.load(args.resume.replace(args.resume.split('/')[-1],''))""",
    ]
    new_sampler_load = """            train_sampler = mining_sampler(train_dataset, batch_size=args.batch_size, dim=args.dim, save_path=args.save_path)
            # Only reload the mining queue on a true same-dataset resume.
            # --finetune (cross-dataset init) never reaches here with a
            # same-sized queue, so it's skipped automatically via args.resume
            # being empty in that case.
            if args.resume:
                train_sampler.load(args.resume.replace(args.resume.split('/')[-1],''))"""
    found = False
    for variant in old_sampler_variants:
        if variant in content:
            content = content.replace(variant, new_sampler_load)
            found = True
            break
    assert found, "sampler load block not found (check train.py manually around train_sampler.load)"
else:
    print("sampler load block already patched, skipping step 4")

with open('train.py', 'w') as f:
    f.write(content)

import ast
ast.parse(content)
print('train.py patched and syntax OK')

In [ ]:
CVUSA_CHECKPOINT = '/content/pretrained/result/model_best.pth.tar'

!sh run_CVIAN.sh \
  /content/CVDisaster/geolocalization/CVIAN/ \
  /content/result_cvian \
  --finetune "$CVUSA_CHECKPOINT"

In [ ]:
import shutil, os

SRC = '/content/result_cvian'
DST = '/content/drive/MyDrive/TransGeo_CVIAN_final'
os.makedirs(DST, exist_ok=True)

for fname in ['checkpoint.pth.tar', 'model_best.pth.tar']:
    src_path = os.path.join(SRC, fname)
    if os.path.isfile(src_path):
        shutil.copy(src_path, os.path.join(DST, fname))
        print(f'Copied {fname} -> {DST}')
    else:
        print(f'WARNING: {fname} not found at {src_path}')